# Practice 06 — RAG & Agents (no API key needed)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theDocWho/lbv-notebooks/blob/main/ai-ml/06-llm-rag-agents.ipynb) [![Open in Kaggle](https://img.shields.io/badge/Kaggle-open-20BEFF?logo=kaggle&logoColor=white)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/theDocWho/lbv-notebooks/main/ai-ml/06-llm-rag-agents.ipynb)

Companion drills for **Phase 6** of [Learn by Visualization](https://learn-by-visuallization.org/illustrated/index.html) —
pairs with the [RAG pipeline](https://learn-by-visuallization.org/illustrated/6-llms-rag-agents/rag-pipeline.html),
[vector search](https://learn-by-visuallization.org/illustrated/6-llms-rag-agents/vector-search.html) and
[ReAct agent](https://learn-by-visuallization.org/illustrated/6-llms-rag-agents/react-agent.html) explainers.
Everything runs locally with TF-IDF embeddings — the *architecture* is the lesson, not the model.

**How this works:** each exercise is a `# TODO` scaffold followed by a self-check cell. Fill in the TODO, re-run both cells, and turn the ❌ into a ✅. No answers are hidden in the notebook — the checks themselves tell you what "correct" means. Runs on local Jupyter, Google Colab, or Kaggle (free, no setup, no GPU needed).

In [ ]:
NEEDED = [("numpy", "numpy"), ("sklearn", "scikit-learn")]

# ---- setup: run me first ----
import importlib.util, subprocess, sys
for pkg, pipname in NEEDED:
    if importlib.util.find_spec(pkg) is None:
        print(f"installing {pipname} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pipname])

RESULTS = {}
def check(name, fn, hint=""):
    """Self-check: PASS/FAIL with a hint. Never raises -- rerun the cell after editing your answer."""
    try:
        ok = bool(fn())
    except Exception as e:
        ok, hint = False, f"{hint} (your code raised {type(e).__name__}: {e})"
    RESULTS[name] = ok
    print(("\u2705 PASS" if ok else "\u274c FAIL") + f" \u2014 {name}" + ("" if ok else f"\n   hint: {hint}"))

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


### Exercise 1 — chunking
Split `doc` into overlapping chunks: `chunk(text, size=8, overlap=3)` → list of strings of ≤8 words,
each chunk starting 5 (= size − overlap) words after the previous. Overlap is what stops answers
from being cut in half at a boundary.

*Hint: step through `words[i : i+size]` with `i += size - overlap`.*

In [ ]:
doc = ("retrieval augmented generation grounds a language model in your documents "
       "the retriever finds relevant chunks the generator writes an answer citing them "
       "if retrieval fails the model hallucinates confidently")

def chunk(text, size=8, overlap=3):
    # TODO: list of space-joined word windows
    ...


In [ ]:
_c = chunk(doc)
check("chunks are ≤ size words", lambda: all(len(c.split()) <= 8 for c in _c), "slice words[i:i+size]")
check("consecutive chunks overlap by 3 words",
      lambda: _c[0].split()[-3:] == _c[1].split()[:3],
      "advance i by size - overlap = 5, so the last 3 words repeat")
check("no text is lost",
      lambda: set(doc.split()) == set(" ".join(_c).split()), "keep stepping until i >= len(words)")

### Exercise 2 — the retriever
Build `retrieve(query, chunks, k=2)`: TF-IDF-embed the chunks, embed the query, return the top-k
chunks by cosine similarity (best first). This is the R in RAG.

*Hint: `TfidfVectorizer().fit(chunks)`; similarities `cosine_similarity(qv, cv)[0]`; `argsort()[::-1][:k]`.*

In [ ]:
def retrieve(query, chunks, k=2):
    # TODO: top-k most similar chunks, best first
    ...


In [ ]:
_chunks = chunk(doc)
_hits = retrieve("what happens when retrieval fails", _chunks)
check("returns k chunks", lambda: len(_hits) == 2, "slice the sorted indices")
check("top hit mentions hallucination",
      lambda: "hallucinates" in _hits[0] or "fails" in _hits[0],
      "the chunk about failing retrieval shares the most rare terms with the query")

### Exercise 3 — grounded prompt assembly
Write `build_prompt(query, hits)` that produces the classic grounded-generation prompt:
context sections labeled `[1]`, `[2]`…, then the question, then an instruction to answer **only**
from the context and cite sources like `[1]`.

*Hint: the checks only require: every hit present, each preceded by its `[n]` label, the query present,
and the words "only" + "cite" somewhere in your instruction.*

In [ ]:
def build_prompt(query, hits):
    # TODO: return the prompt string
    ...


In [ ]:
_p = build_prompt("what happens when retrieval fails", _hits)
check("all retrieved chunks are in the prompt", lambda: all(h in _p for h in _hits), "paste each hit in")
check("chunks are labeled [1], [2]...", lambda: "[1]" in _p and "[2]" in _p, "number the context blocks")
check("the question is included", lambda: "retrieval fails" in _p, "the model needs the query too")
check("instruction demands grounding + citations",
      lambda: "only" in _p.lower() and "cite" in _p.lower(),
      "'answer ONLY from the context above and CITE sources' — the sentence that fights hallucination")

### Exercise 4 — a ReAct loop with a mock tool
`llm()` below is a scripted stand-in for a model that follows the Thought → Action → Observation
protocol. Finish the **agent loop**: call `llm`, parse its `Action: calc[...]`, run the tool, feed
`Observation: ...` back, and stop at `Final Answer:`. Return the final answer string.

*Hint: the loop is: history → llm → if 'Final Answer:' return it → else run tool, append observation.*

In [ ]:
def calc(expr):
    return str(eval(expr, {"__builtins__": {}}, {}))

_script = iter([
    "Thought: I need the total. Action: calc[17 * 23]",
    "Thought: now add tax. Action: calc[391 * 1.1]",
    "Final Answer: 430.1",
])
def llm(history):
    return next(_script)

def run_agent(question, max_turns=5):
    history = question
    # TODO: the ReAct loop
    ...


In [ ]:
check("agent loops tools and returns the final answer",
      lambda: "430.1" in str(run_agent("price of 17 items at 23 plus 10% tax?")),
      "parse the text between 'calc[' and ']', run calc(), append 'Observation: <result>' to history")

### Stretch goal — break your own RAG
Query something NOT in the doc ("who won the 1998 world cup?") and look at what retrieve() returns:
plausible-looking but irrelevant chunks. Write the guard you'd add (similarity threshold? 'I don't
know' instruction?). This failure mode is the whole
[prompt-injection & guardrails](https://learn-by-visuallization.org/illustrated/6b-llm-indepth/prompt-injection.html) story.

In [ ]:
# stretch — your playground


In [ ]:
# ---- self-score ----
passed, total = sum(RESULTS.values()), len(RESULTS)
print(f"Self-score: {passed}/{total} checks passing")
print("\U0001f389 All green \u2014 move on to the next explainer!" if total and passed == total
      else "Rerun each check cell as you fill in the TODOs above.")
